# 08 — Phân tích lỗi CNN và xuất hình báo cáo

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_CNN_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_CNN/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('CNN', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'CNN', 'test_final/metrics_test.json')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import display, Image
from src.CNN.data import CLASSES, CANDIDATES, load_config, configure_gpu, seed_everything
configure_gpu()
seed_everything(42)

## 1. Bảng cặp nhầm và chọn ảnh
Ưu tiên chiều nhầm phổ biến, chênh lệch VND lớn, sai có gap lớn/nhỏ, ảnh đúng đối chứng. Giữ quy tắc cố định sau khi khóa.

In [ ]:
from src.CNN.analysis import analyze_errors
cases, directed = analyze_errors(DATA_ROOT, RESULTS_ROOT)
display(directed)
display(cases[['case_id','relative_path','y_true_vnd','y_pred_vnd','selection_reason']])

## 2. Ảnh lỗi và Grad-CAM
Mỗi hàng: RGB test → Grad-CAM lớp dự đoán → Grad-CAM lớp thật → softmax.
Gradient tính từ logit trước softmax tại Conv cuối. Heatmap0 được ghi rõ, không tự gán nguyên nhân lỗi.
Điền quan sát vào visual_review_notes.csv sau khi xem ảnh.

In [ ]:
for path in sorted((RESULTS_ROOT/'error_analysis').glob('CNN_GRADCAM_*.png')):
    display(Image(filename=str(path)))
display(pd.read_csv(RESULTS_ROOT/'error_analysis/gradcam_checks.csv'))

## 3. Đóng gói kết quả
ZIP gồm checkpoint.keras, lịch sử, khóa, CSV, PNG/PDF và Markdown. Tải trong Output của version đã chạy thành công.

In [ ]:
import shutil
archive = shutil.make_archive(str(RESULTS_ROOT.parent/'CS231_CNN_RESULTS'), 'zip', root_dir=RESULTS_ROOT.parent, base_dir=RESULTS_ROOT.name)
print('Tải từ Output:', archive)